# Notebook 9 - Paper Evidence, Tables, and Figures

This notebook regenerates the Week 9 paper artifacts from committed Week 8 evidence. It does not rerun training or inference, and it requires no GPU. Weak and negative results remain in the generated outputs.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

IN_COLAB = bool(os.environ.get('COLAB_RELEASE_TAG'))
REPO = Path('/content/shepherd-ai') if IN_COLAB else Path.cwd()
if IN_COLAB and not (REPO / 'pyproject.toml').exists():
    subprocess.run(['git', 'clone', '--branch', 'codex/evidence-aware-paper', 'https://github.com/cyberuniversal/shepherd-ai.git', str(REPO)], check=True)
elif IN_COLAB:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'codex/evidence-aware-paper'], check=True)
if not (REPO / 'pyproject.toml').exists():
    raise FileNotFoundError('Open this notebook from the Shepherd-AI repository.')
%cd {REPO}
%pip install -q -e .
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


## 1. Validate registered evidence

The loader refuses incomplete Week 8 audits, missing roadmap metrics, missing source artifacts, non-finite values, and unsupported physical-flight, safety-guarantee, or novelty flags.

In [ ]:
from shepherd_ai.week9_paper import load_week9_paper_evidence
evidence = load_week9_paper_evidence(REPO)
print(json.dumps({
    'week8_decision': evidence['week8_decision'],
    'metric_count': len(evidence['metric_rows']),
    'claim_limits': evidence['claim_limits'],
}, indent=2))


## 2. Regenerate paper artifacts

In [ ]:
subprocess.run(['python', 'scripts/build_week9_paper_artifacts.py'], check=True)


## 3. Inspect the five roadmap metrics

In [ ]:
import csv
from IPython.display import Markdown, display

metric_path = REPO / 'outputs/tables/week9_end_to_end_metrics.csv'
with metric_path.open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))
table = ['| Metric | Value | Denominator | Unit |', '|---|---:|---:|---|']
table.extend(f"| {row['metric']} | {float(row['value']):.6g} | {row['denominator']} | {row['unit']} |" for row in rows)
display(Markdown('\n'.join(table)))


## 4. Inspect measured runtime

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(REPO / 'reports/figures/week9_stage_runtime.png')))


## 5. Review traceability before using paper numbers

In [ ]:
display(Markdown((REPO / 'reports/week9_evidence_traceability.md').read_text(encoding='utf-8')))


## 6. Package shareable paper artifacts

This archive contains no private imagery, audio, credentials, or model weights.

In [ ]:
import zipfile

bundle = REPO / 'week9_paper_artifacts.zip'
files = [
    'outputs/evaluations/week9_paper_evidence.json',
    'outputs/tables/week9_end_to_end_metrics.csv',
    'outputs/tables/week9_runtime_stages.csv',
    'reports/figures/week9_system_architecture.mmd',
    'reports/figures/week9_evaluation_workflow.mmd',
    'reports/figures/week9_stage_runtime.png',
    'reports/week9_evidence_traceability.md',
    'reports/week9_bibliography.md',
    'reports/shepherd_ai_paper_draft.md',
]
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for relative in files:
        path = REPO / relative
        if not path.is_file():
            raise FileNotFoundError(relative)
        archive.write(path, arcname=relative)
print(bundle)


## 7. Build the label-separated monolithic baseline packet

The model input file excludes gold and Shepherd decisions. The separate gold file is used only after inference.

In [ ]:
subprocess.run(['python', 'scripts/evaluate_evidence_aware_decisions.py'], check=True)
subprocess.run(['python', 'scripts/build_monolithic_decision_packet.py'], check=True)
manifest = json.loads((REPO / 'outputs/evaluations/week9_monolithic_diagnostic_manifest_v1.json').read_text(encoding='utf-8'))
print(json.dumps({
    'case_count': manifest['case_count'],
    'gold_decision_counts': manifest['gold_decision_counts'],
    'inputs_contain_gold': manifest['inputs']['contains_gold_labels'],
    'prompt_version': manifest['metadata']['prompt_version'],
}, indent=2))


## 8. Install the frozen T4 inference environment

This is inference only. The runner rejects CPU execution and records the exact installed versions and GPU name.

In [ ]:
if not IN_COLAB:
    raise RuntimeError('Run the monolithic 7B baseline in Google Colab, not on local CPU.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], check=True, capture_output=True, text=True).stdout.strip()
if 'T4' not in gpu:
    raise RuntimeError(f'Expected a T4 runtime, found: {gpu}')
%pip install -q "transformers>=4.48,<5" "accelerate>=1.2,<2" "bitsandbytes>=0.45,<1" "huggingface_hub>=0.27,<1"
print(gpu)


## 9. Run the monolithic Qwen2.5-7B baseline

Raw responses are checkpointed after every case. Add `--resume` to the command after a runtime interruption.

In [ ]:
subprocess.run([
    'python', 'scripts/run_hf_monolithic_decision_baseline.py',
    '--required-device-substring', 'T4',
    '--precision', '4bit',
    '--temperature', '0',
    '--repetitions', '1',
    '--seed', '17',
], check=True)


## 10. Score only after raw inference completes

Invalid JSON remains an error. This diagnostic reuses development evidence and is not the future fresh human-held-out result.

In [ ]:
subprocess.run(['python', 'scripts/evaluate_monolithic_decision_baseline.py'], check=True)
display(Markdown((REPO / 'reports/week9_monolithic_qwen25_7b_evaluation_v1.md').read_text(encoding='utf-8')))
